# Brent Oil Forecasting Notebook 1
## Nickel-Style Baseline Reproduction

## 이 노트북이 답하는 질문
- 메인 니켈 예측 구조를 유가에 옮기면 어느 정도의 기준 성능이 나오는가?
- 여기서 1단계 기본 예측식은 직전 두 시점으로 다음 값을 외삽하는 `Two-Point Linear Extrapolation`이며, 식은 `yhat_t = 2*y_(t-1) - y_(t-2)`이다.
- 타깃 `y`는 `Com_BrentCrudeOil`이고, 입력 `X`는 타깃을 제외한 전체 변수를 `1주 lag`한 값이다. 즉 `X_{t-1}`로 `y_t`를 예측한다.
- 분할은 `Train=644주`, `Validation=12주`, `Test=12주`이며, hybrid weight는 validation으로만 결정하고 test는 최종 보고용으로만 사용했다.
- clean 재실행 결과 best model은 `Hybrid_TwoPointLinear0.7_GB0.3`이며 test RMSE는 `1.3440`이다. 니켈 과제 원형에 가장 가까운 `Hybrid_TwoPointLinear0.8_GB0.2`도 `1.3474`로 거의 동일했다.
- 따라서 유가에서도 `직전 2시점 선형 외삽 + tree 보정`이 강한 재현 기준선으로 작동했다.

## 데이터 흐름 다이어그램
```text
[Weekly market data]
   -> [Target y = Brent oil]
   -> [All X shifted by 1 week]
   -> [Train / Validation / Test split]
   -> [Two-point linear extrapolation + GradientBoosting]
   -> [Validation picks hybrid weight]
   -> [Untouched Test report]
```

설명: 이 노트북은 `니켈 메인 구조가 유가에도 baseline으로 작동하는가`를 확인하는 문서다. 다른 계열과의 직접 비교는 보고서의 추가 점검용 반복 평가에서 따로 수행한다.


## 설정과 결과
- 모델: `GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=3)`
- 기본 외삽식: `yhat_t = 2*y_(t-1) - y_(t-2)`
- 결측치 처리: train median imputation
- hybrid weight 후보: `0.7`, `0.8`, `0.9`

### 결과표

| Model                           |   RMSE |    MAE |   MAPE(%) |   Validation_RMSE |
|:--------------------------------|-------:|-------:|----------:|------------------:|
| Hybrid_TwoPointLinear0.7_GB0.3  | 1.3440 | 1.1317 |    1.8162 |            2.3013 |
| Hybrid_TwoPointLinear0.8_GB0.2  | 1.3474 | 1.1522 |    1.8476 |            2.4552 |
| Hybrid_TwoPointLinear0.9_GB0.1  | 1.4056 | 1.2056 |    1.9305 |            2.6261 |
| TwoPointLinear                  | 1.5125 | 1.2740 |    2.0383 |               nan |
| GradientBoosting                | 2.4451 | 2.1321 |    3.4385 |               nan |

### 상위 feature importance

| feature         |   importance |
|:----------------|-------------:|
| Com_CrudeOil    |       0.9507 |
| Com_Gasoline    |       0.0181 |
| Bonds_KOR_1Y    |       0.0023 |
| Com_LME_Index   |       0.0017 |
| Idx_SnPVIX      |       0.0017 |
| Bonds_IND_10Y   |       0.0014 |
| Idx_Shanghai    |       0.0013 |
| Com_LME_Al_Cash |       0.0010 |
| Bonds_US_2Y     |       0.0010 |
| EX_AUD_USD      |       0.0010 |

설명: `Com_CrudeOil`, `Com_Gasoline` 같은 인접 에너지 시장 변수의 lagged 정보가 큰 역할을 했고, 그 외 일부 금리·환율 변수가 약한 보조 신호로 작동했다.


In [1]:
"""
Direct oil adaptation of the nickel-style hybrid baseline.

Pipeline
1. One-week lag all exogenous variables to prevent contemporaneous leakage.
2. Fit a GradientBoostingRegressor on the training split.
3. Build a nickel-style hybrid forecast:
       y_hat = w * TwoPointLinear + (1 - w) * GB
4. Pick the best weight on validation and report untouched test metrics.
"""

from __future__ import annotations

import json
import math
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error


DATA_PATH = "data_weekly_260120.csv"
TARGET = "Com_BrentCrudeOil"
VAL_START = "2025-08-04"
TEST_START = "2025-10-27"
OUT_DIR = "output_oil_nickel_style"
RANDOM_STATE = 42
HYBRID_WEIGHTS = [0.70, 0.80, 0.90]


@dataclass
class SplitData:
    X_train: pd.DataFrame
    X_val: pd.DataFrame
    X_test: pd.DataFrame
    y_train: pd.Series
    y_val: pd.Series
    y_test: pd.Series
    two_point_val: np.ndarray
    two_point_test: np.ndarray


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return math.sqrt(mean_squared_error(y_true, y_pred))


def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.clip(np.abs(y_true), 1e-8, None)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def load_frame() -> pd.DataFrame:
    df = pd.read_csv(DATA_PATH)
    df["dt"] = pd.to_datetime(df["dt"])
    df = df.sort_values("dt").set_index("dt")
    df.index.freq = "W-MON"
    return df


def calc_two_point_linear(y: pd.Series, indices: pd.Index) -> np.ndarray:
    preds = []
    for idx in indices:
        loc = y.index.get_loc(idx)
        prev_1 = float(y.iloc[loc - 1])
        prev_2 = float(y.iloc[loc - 2])
        preds.append(prev_1 + (prev_1 - prev_2))
    return np.asarray(preds, dtype=float)


def prepare_splits(df: pd.DataFrame) -> SplitData:
    feature_cols = [c for c in df.columns if c != TARGET]
    X_all = df[feature_cols].shift(1)
    y_all = df[TARGET].copy()

    mask_train = df.index < VAL_START
    mask_val = (df.index >= VAL_START) & (df.index < TEST_START)
    mask_test = df.index >= TEST_START

    X_train = X_all.loc[mask_train]
    X_val = X_all.loc[mask_val]
    X_test = X_all.loc[mask_test]
    y_train = y_all.loc[mask_train]
    y_val = y_all.loc[mask_val]
    y_test = y_all.loc[mask_test]

    two_point_val = calc_two_point_linear(y_all, y_val.index)
    two_point_test = calc_two_point_linear(y_all, y_test.index)

    return SplitData(
        X_train=X_train,
        X_val=X_val,
        X_test=X_test,
        y_train=y_train,
        y_val=y_val,
        y_test=y_test,
        two_point_val=two_point_val,
        two_point_test=two_point_test,
    )


def fit_gb(split: SplitData) -> tuple[GradientBoostingRegressor, np.ndarray, np.ndarray]:
    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(split.X_train)
    X_val = imputer.transform(split.X_val)
    X_test = imputer.transform(split.X_test)

    model = GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, split.y_train)
    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    return model, val_pred, test_pred


def build_result(model_name: str, y_true: pd.Series, pred: np.ndarray) -> dict[str, float | str]:
    yt = y_true.to_numpy(dtype=float)
    return {
        "Model": model_name,
        "RMSE": rmse(yt, pred),
        "MAE": float(mean_absolute_error(yt, pred)),
        "MAPE(%)": mape(yt, pred),
    }

## 실행 코드
아래 셀은 기준선 실험 전체를 재현한다. 교수님이 코드 검토 시 확인해야 할 핵심은 `shift(1)`, `validation-based weight selection`, `test untouched` 세 가지다.

In [2]:
def main() -> None:
    os.makedirs(OUT_DIR, exist_ok=True)

    df = load_frame()
    split = prepare_splits(df)
    model, gb_val, gb_test = fit_gb(split)

    results = []

    results.append(build_result("TwoPointLinear", split.y_test, split.two_point_test))
    results.append(build_result("GradientBoosting", split.y_test, gb_test))

    best_weight = None
    best_val_rmse = float("inf")
    best_val_pred = None
    best_test_pred = None

    for w in HYBRID_WEIGHTS:
        val_pred = w * split.two_point_val + (1.0 - w) * gb_val
        test_pred = w * split.two_point_test + (1.0 - w) * gb_test
        val_rmse = rmse(split.y_val.to_numpy(dtype=float), val_pred)
        label = f"Hybrid_TwoPointLinear{w:.1f}_GB{1.0-w:.1f}"
        row = build_result(label, split.y_test, test_pred)
        row["Validation_RMSE"] = val_rmse
        results.append(row)

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_weight = w
            best_val_pred = val_pred
            best_test_pred = test_pred

    results_df = pd.DataFrame(results)
    results_df.to_csv(f"{OUT_DIR}/results.csv", index=False)

    feature_importance = (
        pd.DataFrame(
            {
                "feature": split.X_train.columns,
                "importance": model.feature_importances_,
            }
        )
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    feature_importance.to_csv(f"{OUT_DIR}/feature_importance.csv", index=False)

    if best_weight is None or best_val_pred is None or best_test_pred is None:
        raise RuntimeError("Failed to choose a hybrid weight.")

    pd.DataFrame(
        {
            "dt": split.y_test.index,
            "actual": split.y_test.to_numpy(dtype=float),
            "two_point_linear": split.two_point_test,
            "gradient_boosting": gb_test,
            f"hybrid_{best_weight:.2f}": best_test_pred,
        }
    ).to_csv(f"{OUT_DIR}/test_predictions.csv", index=False)

    config = {
        "target": TARGET,
        "split": {
            "train_end": str(split.y_train.index[-1].date()),
            "val_start": VAL_START,
            "val_end": str(split.y_val.index[-1].date()),
            "test_start": TEST_START,
            "test_end": str(split.y_test.index[-1].date()),
        },
        "n_features": int(split.X_train.shape[1]),
        "best_weight_by_validation": best_weight,
        "best_validation_rmse": best_val_rmse,
    }
    with open(f"{OUT_DIR}/config.json", "w", encoding="utf-8") as fp:
        json.dump(config, fp, indent=2, ensure_ascii=False)

    print("=" * 72)
    print("Oil Nickel-Style Hybrid Baseline")
    print("=" * 72)
    print(f"Target          : {TARGET}")
    print(f"Train / Val / Test: {len(split.y_train)} / {len(split.y_val)} / {len(split.y_test)}")
    print(f"Best hybrid w   : {best_weight:.2f}")
    print(f"Best val RMSE   : {best_val_rmse:.4f}")
    print()
    print(results_df.sort_values("RMSE").to_string(index=False))
    print()
    print(f"Output directory: {OUT_DIR}")


if __name__ == "__main__":
    main()

Oil Nickel-Style Hybrid Baseline
Target          : Com_BrentCrudeOil
Train / Val / Test: 644 / 12 / 12
Best hybrid w   : 0.70
Best val RMSE   : 2.3013

                         Model     RMSE      MAE  MAPE(%)  Validation_RMSE
Hybrid_TwoPointLinear0.7_GB0.3 1.343974 1.131665 1.816181         2.301291
Hybrid_TwoPointLinear0.8_GB0.2 1.347368 1.152168 1.847581         2.455236
Hybrid_TwoPointLinear0.9_GB0.1 1.405645 1.205603 1.930500         2.626144
                TwoPointLinear 1.512472 1.274002 2.038304              NaN
              GradientBoosting 2.445133 2.132102 3.438467              NaN

Output directory: output_oil_nickel_style


## 해석
- 니켈 메인 방식은 유가에도 그대로 이식 가능했다.
- `GradientBoosting` 단독은 약했지만, `Two-Point Linear Extrapolation`과 결합하면 안정적으로 개선됐다.
- 따라서 이 노트북은 advanced 실험의 출발점이 되는 `재현 기준선`으로 해석하면 된다.
